# FitBot LangChain AI Agent

FitBot is an AI personal trainer prototype. It takes daily activity data like steps, active minutes, and sedentary minutes, predicts calories burned, and recommends workouts based on the user’s goal, equipment, experience level, and limitations. I designed it to work even without an API key, but it is also ready to connect to LangChain for LLM tool-calling.

## Cell 1 —  Installation

“This cell is here to make the notebook portable. If the environment does not already have the required libraries, I can uncomment this line and install everything needed to run the project.”

This cell lists the Python packages needed for the project. It is commented out because many environments, including some Colab sessions, may already have several of these libraries installed. If a library is missing, the user can uncomment the install line and run it.

The packages include data science tools like Pandas, NumPy, and Scikit-learn, plus LangChain for agent/tool functionality and Gradio for the optional user interface.

In [1]:
# install for Colab
# !pip install -q pandas numpy scikit-learn langchain langchain-openai gradio


## Cell 2 — Import Libraries and Set Random Seeds

“This cell prepares the notebook by importing the tools I need. It also sets random seeds so the project is reproducible instead of changing randomly every time I run it.”

This cell imports every library used in the notebook. Pandas and NumPy handle data processing, Scikit-learn handles machine learning, and Python dataclasses help create a simple memory system for the AI agent.

The cell also sets random seeds for Python and NumPy. This makes the results more reproducible, which means the notebook should create the same fallback data and similar model results each time it is run.

In [2]:
import os, random, math
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Any
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
random.seed(42)
np.random.seed(42)


## Cell 3 — Data Loading Section

“This section begins the data pipeline. The notebook first looks for the real Fitbit data file, but if that file is missing, it creates a realistic sample dataset with the same type of columns.”

This markdown section introduces the data loading step. The project is designed to load the real Fitbit/Fitabase dataset if the CSV file is available. If the file is not available, the notebook automatically creates a fallback Fitbit-style dataset so the project can still run from beginning to end.

This is important because final projects are often graded in a new environment where the original Kaggle file may not already be uploaded. The fallback dataset prevents the code from breaking.

## Cell 4 — Load the Fitbit Dataset or Create Fallback Data

“This cell is responsible for getting the dataset ready. It either loads the real Kaggle Fitbit data or creates a fallback dataset with realistic activity patterns. This makes the project reliable because it can still run even if the CSV file is missing.”

This code defines the `load_fitbit_data()` function. The function first checks whether `dailyActivity_merged.csv` exists in the notebook environment. If the file exists, it loads the real Fitbit/Fitabase dataset using Pandas.

If the file does not exist, the function creates a synthetic fallback dataset with 940 records and 31 simulated users. The fallback data includes realistic fitness-tracking columns such as steps, active minutes, sedentary minutes, and calories. Calories are calculated using a formula that combines baseline calories, steps, active minutes, and random noise.

At the end of the cell, the function is called and the first few rows of the dataset are displayed with `raw_df.head()`. This lets us verify that the data loaded correctly.

In [3]:
def load_fitbit_data(path='dailyActivity_merged.csv'):
    if os.path.exists(path):
        print('Loaded Kaggle/Fitbit dataset')
        return pd.read_csv(path)
    print('Kaggle file not found. Creating fallback Fitbit-style dataset.')
    n = 940
    dates = pd.date_range('2016-04-12', periods=31)
    df = pd.DataFrame({
        'Id': np.random.choice(range(100000, 100031), n),
        'ActivityDate': np.random.choice(dates.astype(str), n),
        'TotalSteps': np.clip(np.random.normal(7600, 3500, n), 0, 19000).astype(int),
        'VeryActiveMinutes': np.clip(np.random.normal(20, 18, n), 0, 120).astype(int),
        'FairlyActiveMinutes': np.clip(np.random.normal(15, 12, n), 0, 90).astype(int),
        'LightlyActiveMinutes': np.clip(np.random.normal(185, 75, n), 20, 500).astype(int),
        'SedentaryMinutes': np.clip(np.random.normal(930, 160, n), 400, 1300).astype(int),
    })
    active_factor = df['VeryActiveMinutes']*8 + df['FairlyActiveMinutes']*5 + df['LightlyActiveMinutes']*2
    df['Calories'] = (1450 + df['TotalSteps']*.055 + active_factor + np.random.normal(0, 110, n)).clip(1200, 3800).astype(int)
    return df
raw_df = load_fitbit_data()
raw_df.head()


Kaggle file not found. Creating fallback Fitbit-style dataset.


,Id,ActivityDate,TotalSteps,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories
0,100006,2016-04-17,9726,0,16,176,1104,2261
1,100019,2016-04-23,2282,42,14,259,1121,2618
2,100028,2016-05-05,7581,18,0,327,755,2647
3,100014,2016-05-09,5294,42,28,241,1057,2604
4,100010,2016-05-02,1686,20,5,220,900,1950


## Cell 5 — Preprocessing Section
“This section prepares the raw dataset for machine learning. Cleaning the data is important because messy data can lead to inaccurate predictions and unreliable recommendations.”

This section introduces the data cleaning process. Raw datasets often contain missing values, duplicate rows, incorrect data types, and unnecessary columns. Before training a model, we need to clean the dataset so the machine learning pipeline receives consistent and reliable input.

The preprocessing step keeps only the core fitness columns, converts dates and numbers into proper formats, fills missing values, and removes duplicates.

## Cell 6 — Clean the Dataset and Report Data Quality

“This cell cleans the data. It fixes column formatting, converts dates and numbers, fills missing values with medians, removes duplicate rows, and then gives me a quality report so I can prove the data improved before modeling.”

This code defines the `preprocess_data()` function. First, it makes a copy of the dataset so the original raw data is not changed. Then it standardizes column names by removing extra spaces and keeps only the columns needed for the project.

The function converts the `ActivityDate` column into a datetime format so we can later create day-of-week features. It also converts all numeric columns into actual numeric values and fills missing numeric values using the median. Median imputation is useful because it is less affected by extreme outliers than the mean.

The function also counts missing values and duplicate rows before and after cleaning. This creates a small quality report showing whether preprocessing improved the dataset. Finally, the cleaned dataset and the quality report are returned.

In [4]:
CORE_COLUMNS = ['Id','ActivityDate','TotalSteps','VeryActiveMinutes','FairlyActiveMinutes','LightlyActiveMinutes','SedentaryMinutes','Calories']
def preprocess_data(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]
    df = df[[c for c in CORE_COLUMNS if c in df.columns]]
    df['ActivityDate'] = pd.to_datetime(df['ActivityDate'], errors='coerce')
    missing_before = int(df.isna().sum().sum()); duplicates_before = int(df.duplicated().sum())
    for col in df.columns:
        if col != 'ActivityDate':
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].median())
    df['ActivityDate'] = df['ActivityDate'].fillna(df['ActivityDate'].mode()[0])
    df = df.drop_duplicates().reset_index(drop=True)
    return df, {'missing_before': missing_before, 'missing_after': int(df.isna().sum().sum()), 'duplicates_before': duplicates_before, 'duplicates_after': int(df.duplicated().sum()), 'processed_rows': len(df)}
clean_df, quality_report = preprocess_data(raw_df)
quality_report


{'missing_before': 0,
 'missing_after': 0,
 'duplicates_before': 0,
 'duplicates_after': 0,
 'processed_rows': 940}

## Cell 7 — Feature Engineering Section

“Feature engineering is where I turn basic tracker numbers into more meaningful fitness signals. These new features help the AI understand the user better than raw steps alone.”

This section introduces feature engineering, which means creating new useful columns from the original raw data. Instead of only using basic columns like total steps, FitBot creates additional activity indicators that better describe a user’s fitness behavior.

These new features help the model and recommendation engine understand activity quality, sedentary behavior, goal progress, and recovery needs.

## Cell 8 — Create Fitness-Based Engineered Features

“This cell creates the intelligence behind the project. It turns raw Fitbit data into features like activity score, recovery flag, step goal percentage, and intensity category so FitBot can make smarter predictions and recommendations.”

This code defines the `engineer_features()` function. It creates several new columns that make the dataset more useful for both prediction and recommendations.

`DayOfWeek` identifies the day name from the activity date, and `WeekendFlag` marks whether the day was a weekend. `TotalActiveMinutes` combines very active, fairly active, and lightly active minutes into one total movement measure. `SedentaryRatio` calculates how much of the day was sedentary compared to total recorded movement time.

`StepGoalPct` measures progress toward a 10,000-step benchmark, while `StepGap` calculates how many steps remain before reaching that goal. `ActivityScore` combines steps, active minutes, and sedentary behavior into one weighted fitness score. `RecoveryFlag` identifies days where the user may need lighter activity because steps are low and sedentary time is high. Finally, `IntensityCategory` and `StepBand` turn numerical patterns into readable categories.

In [5]:
def engineer_features(df):
    df = df.copy()
    df['DayOfWeek'] = df['ActivityDate'].dt.day_name()
    df['WeekendFlag'] = df['DayOfWeek'].isin(['Saturday','Sunday']).astype(int)
    df['TotalActiveMinutes'] = df['VeryActiveMinutes'] + df['FairlyActiveMinutes'] + df['LightlyActiveMinutes']
    df['SedentaryRatio'] = df['SedentaryMinutes'] / (df['SedentaryMinutes'] + df['TotalActiveMinutes'] + 1)
    df['StepGoalPct'] = np.minimum(df['TotalSteps'] / 10000, 1.5)
    df['StepGap'] = np.maximum(10000 - df['TotalSteps'], 0)
    df['ActivityScore'] = (df['TotalSteps']/1000) + (df['VeryActiveMinutes']*.25) + (df['FairlyActiveMinutes']*.15) + (df['LightlyActiveMinutes']*.03) - (df['SedentaryRatio']*4)
    df['RecoveryFlag'] = ((df['TotalSteps'] < 3500) & (df['SedentaryRatio'] > .82)).astype(int)
    df['IntensityCategory'] = pd.cut(df['ActivityScore'], [-999,5,11,999], labels=['low','moderate','high']).astype(str)
    df['StepBand'] = pd.cut(df['TotalSteps'], [-1,5000,10000,15000,999999], labels=['low','moderate','high','very_high']).astype(str)
    return df
feature_df = engineer_features(clean_df)
feature_df.head()


,Id,ActivityDate,TotalSteps,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,DayOfWeek,WeekendFlag,TotalActiveMinutes,SedentaryRatio,StepGoalPct,StepGap,ActivityScore,RecoveryFlag,IntensityCategory,StepBand
0,100006,2016-04-17,9726,0,16,176,1104,2261,Sunday,1,192,0.851195,0.9726,274,14.001220,0,high,moderate
1,100019,2016-04-23,2282,42,14,259,1121,2618,Saturday,1,315,0.780097,0.2282,7718,19.531610,0,high,low
2,100028,2016-05-05,7581,18,0,327,755,2647,Thursday,0,345,0.685740,0.7581,2419,19.148039,0,high,moderate
3,100014,2016-05-09,5294,42,28,241,1057,2604,Monday,0,311,0.772096,0.5294,4706,24.135614,0,high,moderate
4,100010,2016-05-02,1686,20,5,220,900,1950,Monday,0,245,0.785340,0.1686,8314,10.894639,0,moderate,low


## Cell 9 — Model Training Section

“This section trains the calorie prediction model. I also compare it to a baseline average predictor so I can measure whether my model actually adds value.”

This section introduces the machine learning part of the project. The goal is to train a supervised learning model that predicts calories burned based on the user’s activity data and engineered features.

The model is compared to a simple baseline that predicts the average calorie value. This comparison helps prove whether the machine learning model is actually better than a basic guess.

## Cell 10 — Train and Evaluate the Random Forest Calorie Model

“This cell trains the actual machine learning model. It preprocesses numeric and categorical features, trains a Random Forest Regressor, predicts calories on the test set, and compares the results against a baseline average-calorie prediction.”

This cell selects the input features and target variable for the model. The input features include steps, active minutes, sedentary minutes, engineered features, and categorical labels. The target variable is `Calories`, which is what the model learns to predict.

The cell separates numeric and categorical features. Numeric features are standardized using `StandardScaler`, while categorical features are converted into machine-readable format using `OneHotEncoder`. These transformations are placed inside a `ColumnTransformer`, which allows different preprocessing steps for different column types.

Next, the code creates a Scikit-learn `Pipeline` that combines preprocessing and a `RandomForestRegressor`. A train-test split is used so the model trains on one portion of the data and is evaluated on unseen data. The model is measured using MAE, RMSE, and R-squared.

The baseline predictor simply guesses the average training-set calorie value for every test row. Comparing FitBot to this baseline shows whether the model learned meaningful patterns.

In [6]:
FEATURES = ['TotalSteps','VeryActiveMinutes','FairlyActiveMinutes','LightlyActiveMinutes','SedentaryMinutes','TotalActiveMinutes','SedentaryRatio','StepGoalPct','StepGap','ActivityScore','RecoveryFlag','WeekendFlag','IntensityCategory','StepBand']
X = feature_df[FEATURES]; y = feature_df['Calories']
numeric_features = X.select_dtypes(include=['int64','float64','int32','float32']).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
preprocessor = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
calorie_model = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=150, random_state=42))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
calorie_model.fit(X_train, y_train)
y_pred = calorie_model.predict(X_test)
baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
metrics = {'Baseline MAE': round(mean_absolute_error(y_test, baseline_pred),2), 'FitBot MAE': round(mean_absolute_error(y_test, y_pred),2), 'Baseline RMSE': round(math.sqrt(mean_squared_error(y_test, baseline_pred)),2), 'FitBot RMSE': round(math.sqrt(mean_squared_error(y_test, y_pred)),2), 'FitBot R2': round(r2_score(y_test, y_pred),3)}
metrics


{'Baseline MAE': 234.28,
 'FitBot MAE': 101.4,
 'Baseline RMSE': 290.13,
 'FitBot RMSE': 123.16,
 'FitBot R2': 0.819}

## Cell 11 — Exercise Knowledge Base Section

“This section creates FitBot’s workout knowledge base. It stores exercises in a structured format so the recommendation engine can filter them intelligently.”

This section introduces the exercise database used by FitBot. The project does not rely only on machine learning; it also uses structured workout knowledge to recommend safe and relevant exercises.

Each exercise includes metadata such as goal, equipment type, difficulty level, target muscle group, impact level, limitations to avoid, sets, reps, and rest time.

## Cell 12 — Build the Structured Exercise Knowledge Base

“This cell builds the workout database. Each exercise is stored with details that help FitBot decide whether it matches the user’s goal, equipment, experience level, and physical limitations.”

This cell creates the `EXERCISES` list. Each item in the list is a dictionary representing one exercise. The dictionary stores important information such as the exercise name, supported fitness goals, equipment needed, difficulty level, target muscle group, impact level, injury limitations, sets, reps, and rest time.

This structure allows the recommendation engine to filter exercises based on the user’s profile. For example, if a user has knee pain, the system can avoid exercises that list knee pain in `avoid_if`. If a user only has bodyweight equipment, the system avoids dumbbell or gym-only exercises.

In [7]:
EXERCISES = [
    {'name':'Brisk Walk','goals':['lose weight','maintain fitness','recovery'],'equipment':'bodyweight','level':'beginner','muscle':'cardio','impact':'low','avoid_if':[],'sets':'1','reps':'20-30 min','rest':'as needed'},
    {'name':'Bodyweight Squat','goals':['gain strength','maintain fitness','lose weight'],'equipment':'bodyweight','level':'beginner','muscle':'legs','impact':'low','avoid_if':['knee pain'],'sets':'2-3','reps':'10-15','rest':'60 sec'},
    {'name':'Glute Bridge','goals':['gain strength','recovery','maintain fitness'],'equipment':'bodyweight','level':'beginner','muscle':'glutes','impact':'low','avoid_if':[],'sets':'2-3','reps':'12-15','rest':'45 sec'},
    {'name':'Incline Push-Up','goals':['gain strength','maintain fitness','lose weight'],'equipment':'bodyweight','level':'beginner','muscle':'chest','impact':'low','avoid_if':['shoulder pain','wrist pain'],'sets':'2-3','reps':'8-12','rest':'60 sec'},
    {'name':'Forearm Plank','goals':['gain strength','maintain fitness','lose weight'],'equipment':'bodyweight','level':'beginner','muscle':'core','impact':'low','avoid_if':['back pain','shoulder pain'],'sets':'2-3','reps':'20-40 sec','rest':'45 sec'},
    {'name':'Dumbbell Goblet Squat','goals':['gain strength','lose weight','maintain fitness'],'equipment':'dumbbells','level':'beginner','muscle':'legs','impact':'low','avoid_if':['knee pain'],'sets':'3','reps':'8-12','rest':'75 sec'},
    {'name':'Dumbbell Romanian Deadlift','goals':['gain strength','maintain fitness'],'equipment':'dumbbells','level':'beginner','muscle':'hamstrings/glutes','impact':'low','avoid_if':['back pain'],'sets':'3','reps':'8-12','rest':'75 sec'},
    {'name':'One-Arm Dumbbell Row','goals':['gain strength','maintain fitness'],'equipment':'dumbbells','level':'beginner','muscle':'back','impact':'low','avoid_if':[],'sets':'3','reps':'10 each side','rest':'60 sec'},
    {'name':'Dumbbell Floor Press','goals':['gain strength','maintain fitness'],'equipment':'dumbbells','level':'beginner','muscle':'chest','impact':'low','avoid_if':['shoulder pain'],'sets':'3','reps':'8-12','rest':'75 sec'},
    {'name':'Bike Intervals','goals':['lose weight','build endurance'],'equipment':'gym','level':'beginner','muscle':'cardio','impact':'low','avoid_if':[],'sets':'6 rounds','reps':'1 min hard / 1 min easy','rest':'built in'},
    {'name':'Lat Pulldown','goals':['gain strength','maintain fitness'],'equipment':'gym','level':'beginner','muscle':'back','impact':'low','avoid_if':['shoulder pain'],'sets':'3','reps':'8-12','rest':'75 sec'},
    {'name':'Chest Press Machine','goals':['gain strength','maintain fitness'],'equipment':'gym','level':'beginner','muscle':'chest','impact':'low','avoid_if':['shoulder pain'],'sets':'3','reps':'8-12','rest':'75 sec'},
    {'name':'Resistance Band Row','goals':['gain strength','maintain fitness'],'equipment':'resistance bands','level':'beginner','muscle':'back','impact':'low','avoid_if':[],'sets':'2-3','reps':'12-15','rest':'45 sec'},
    {'name':'Band Chest Press','goals':['gain strength','maintain fitness'],'equipment':'resistance bands','level':'beginner','muscle':'chest','impact':'low','avoid_if':['shoulder pain'],'sets':'2-3','reps':'10-15','rest':'45 sec'},
    {'name':'Band Squat','goals':['gain strength','lose weight','maintain fitness'],'equipment':'resistance bands','level':'beginner','muscle':'legs','impact':'low','avoid_if':['knee pain'],'sets':'2-3','reps':'10-15','rest':'60 sec'},
    {'name':'Cat-Cow Stretch','goals':['recovery','maintain fitness'],'equipment':'bodyweight','level':'beginner','muscle':'mobility','impact':'low','avoid_if':[],'sets':'2','reps':'8-10','rest':'none'},
    {'name':'Dead Bug','goals':['recovery','gain strength','maintain fitness'],'equipment':'bodyweight','level':'beginner','muscle':'core','impact':'low','avoid_if':[],'sets':'2-3','reps':'8 each side','rest':'45 sec'},
    {'name':'Bird Dog','goals':['recovery','gain strength','maintain fitness'],'equipment':'bodyweight','level':'beginner','muscle':'core/back','impact':'low','avoid_if':[],'sets':'2-3','reps':'8 each side','rest':'45 sec'}
]
exercise_df = pd.DataFrame(EXERCISES)
exercise_df.head()


,name,goals,equipment,level,muscle,impact,avoid_if,sets,reps,rest
0,Brisk Walk,"[lose weight, maintain fitness, recovery]",bodyweight,beginner,cardio,low,[],1,20-30 min,as needed
1,Bodyweight Squat,"[gain strength, maintain fitness, lose weight]",bodyweight,beginner,legs,low,[knee pain],2-3,10-15,60 sec
2,Glute Bridge,"[gain strength, recovery, maintain fitness]",bodyweight,beginner,glutes,low,[],2-3,12-15,45 sec
3,Incline Push-Up,"[gain strength, maintain fitness, lose weight]",bodyweight,beginner,chest,low,"[shoulder pain, wrist pain]",2-3,8-12,60 sec
4,Forearm Plank,"[gain strength, maintain fitness, lose weight]",bodyweight,beginner,core,low,"[back pain, shoulder pain]",2-3,20-40 sec,45 sec


## Cell 13 — Workout Function Section

“This section is where FitBot starts acting like a personal trainer. It uses the exercise database and user inputs to build a complete personalized workout plan.”

This section introduces the core recommendation functions. These functions connect the exercise knowledge base, user profile information, and calorie prediction model into one workout recommendation system.

The system filters exercises, checks limitations, balances the workout across cardio, strength, and core movements, predicts calories, and returns a complete text-based plan.

## Cell 14 — Filter Exercises, Predict Calories, and Generate Workout Plans

“This cell is the main recommendation engine. It filters exercises based on the user’s needs, avoids unsafe movements, creates a balanced workout, predicts calories with the trained model, and returns a full personalized plan.”

This cell defines the main workout recommendation logic. `LEVEL_RANK` assigns a numeric difficulty ranking to beginner, intermediate, and advanced levels. The `allowed_by_level()` function makes sure the user only receives exercises appropriate for their experience level.

The `limitation_matches()` function checks whether an exercise should be avoided because of user limitations such as knee pain, shoulder pain, wrist pain, or back pain. The `filter_exercises()` function then filters the exercise database by goal, equipment, experience level, limitations, and recovery status.

The `select_balanced_workout()` function chooses a balanced set of exercises. For example, a weight-loss workout may include cardio, legs, upper body, and core. A recovery workout focuses more on mobility and low-impact movement.

The calorie prediction helper builds a one-row input sample using the same engineered features as the training data, then sends it through the trained model. Finally, `recommend_workout()` combines everything into a readable workout plan with estimated calories, activity classification, selected exercises, and safety notes.

In [8]:
LEVEL_RANK = {'beginner':1, 'intermediate':2, 'advanced':3}
def allowed_by_level(exercise_level, user_level): return LEVEL_RANK[exercise_level] <= LEVEL_RANK[user_level]
def limitation_matches(exercise, limitations):
    text = limitations.lower().strip()
    if text in ['none','no','n/a','']: return False
    return any(lim in text for lim in exercise['avoid_if'])
def filter_exercises(goal, equipment, experience_level, limitations, recovery=False):
    available_equipment = [equipment.lower(), 'bodyweight']
    if recovery: goal = 'recovery'
    matches = []
    for ex in EXERCISES:
        if goal not in ex['goals']: continue
        if ex['equipment'] not in available_equipment: continue
        if not allowed_by_level(ex['level'], experience_level): continue
        if limitation_matches(ex, limitations): continue
        matches.append(ex)
    if not matches:
        matches = [ex for ex in EXERCISES if ex['name'] in ['Brisk Walk','Glute Bridge','Dead Bug','Bird Dog','Cat-Cow Stretch']]
    return matches
def select_balanced_workout(goal, equipment, experience_level, limitations, steps, active_minutes):
    recovery = steps < 3500 or active_minutes < 10 or goal == 'recovery'
    pool = filter_exercises(goal, equipment, experience_level, limitations, recovery)
    cardio = [ex for ex in pool if 'cardio' in ex['muscle']]
    legs = [ex for ex in pool if ex['muscle'] in ['legs','glutes','hamstrings/glutes']]
    upper = [ex for ex in pool if ex['muscle'] in ['chest','back']]
    core = [ex for ex in pool if 'core' in ex['muscle'] or 'mobility' in ex['muscle']]
    selected=[]
    def add_from(group, count):
        random.shuffle(group)
        for ex in group[:count]:
            if ex not in selected: selected.append(ex)
    if recovery: add_from(core,3); add_from(cardio,1)
    elif goal == 'lose weight': add_from(cardio,1); add_from(legs,1); add_from(upper,1); add_from(core,1)
    elif goal == 'gain strength': add_from(legs,2); add_from(upper,2); add_from(core,1)
    elif goal == 'build endurance': add_from(cardio,2); add_from(legs,1); add_from(core,1)
    else: add_from(cardio,1); add_from(legs,1); add_from(upper,1); add_from(core,1)
    for ex in pool:
        if len(selected) >= 5: break
        if ex not in selected: selected.append(ex)
    return selected[:5], recovery
def build_activity_input(steps, active_minutes, sedentary_minutes):
    very = int(active_minutes*.25); fairly = int(active_minutes*.20); lightly = max(int(active_minutes)-very-fairly,0)
    row = pd.DataFrame([{'Id':0, 'ActivityDate':pd.Timestamp.today(), 'TotalSteps':int(steps), 'VeryActiveMinutes':very, 'FairlyActiveMinutes':fairly, 'LightlyActiveMinutes':lightly, 'SedentaryMinutes':int(sedentary_minutes), 'Calories':0}])
    return engineer_features(row)
def predict_calories_from_activity(steps, active_minutes, sedentary_minutes):
    row = build_activity_input(steps, active_minutes, sedentary_minutes)
    return int(calorie_model.predict(row[FEATURES])[0])
def recommend_workout(goal, equipment, experience_level, limitations, steps, active_minutes, sedentary_minutes, workout_minutes):
    activity_row = build_activity_input(steps, active_minutes, sedentary_minutes)
    predicted_calories = int(calorie_model.predict(activity_row[FEATURES])[0])
    selected, recovery = select_balanced_workout(goal, equipment, experience_level, limitations, int(steps), int(active_minutes))
    step_gap = max(10000 - int(steps), 0)
    if recovery:
        title='Recovery Workout Recommendation'; warmup='3-5 minutes easy walking and gentle mobility'; cooldown='5 minutes stretching and slow breathing'
    elif goal == 'lose weight':
        title='Weight Loss Workout Recommendation'; warmup='5 minutes brisk walking or marching in place'; cooldown='5 minutes easy walking and stretching'
    elif goal == 'gain strength':
        title='Strength Workout Recommendation'; warmup='5 minutes mobility and light warm-up sets'; cooldown='5 minutes stretching the muscles trained'
    elif goal == 'build endurance':
        title='Endurance Workout Recommendation'; warmup='5 minutes easy cardio'; cooldown='5 minutes slow walking and breathing'
    else:
        title='Balanced Fitness Workout Recommendation'; warmup='5 minutes easy cardio and mobility'; cooldown='5 minutes full-body stretching'
    response=[title,'',f'Estimated calories burned today: {predicted_calories}',f'Steps: {int(steps):,}',f'Active minutes: {int(active_minutes)}',f'Workout time available: {int(workout_minutes)} minutes','', 'Warm-Up:', f'- {warmup}', '', 'Main Workout:']
    for i, ex in enumerate(selected,1): response.append(f"{i}. {ex['name']} — {ex['sets']} sets, {ex['reps']}, rest {ex['rest']}")
    response += ['', 'Cooldown:', f'- {cooldown}', '']
    response.append(f'Step Guidance: You are {step_gap:,} steps below 10,000. Add a short walk later if you feel good.' if step_gap > 0 else 'Step Guidance: You reached 10,000 steps, so no extra cardio is needed today.')
    response.append('Sedentary Guidance: Add 2-3 short movement breaks during the day.' if int(sedentary_minutes) > 900 else 'Sedentary Guidance: Your sedentary time is manageable today.')
    if limitations.lower() not in ['none','no','n/a','']: response.append(f"Safety Note: Because you listed '{limitations}', avoid painful movements and reduce intensity if needed.")
    response += ['', 'Progression: If this workout feels easy for two sessions, add one set or 5 extra minutes next time.']
    return '\n'.join(response)


## Cell 15 — Memory Agent Section

“This section adds memory. FitBot can save user check-ins and later produce a progress report, which makes it feel more like a real assistant instead of a one-time calculator.”

This section introduces FitBot’s memory system. A basic AI assistant only responds to one prompt at a time, but an agent with memory can track previous check-ins and summarize progress over time.

This memory system works without an API key, which makes it useful as a fallback assistant even when the LangChain LLM agent is not active.

## Cell 16 — Create the User Profile and FitBot Memory Agent

“This cell creates the memory-based version of FitBot. It stores the user’s profile and workout history, generates responses, and can summarize progress after multiple check-ins.”

This cell uses Python dataclasses to create a `UserProfile`. The profile stores the user’s name, goal, equipment, experience level, limitations, and workout history. The history list is important because it stores each check-in over time.

The `FitBotWorkoutAgent` class manages the user profile and responses. The `update_profile()` method saves the latest goal and preferences. The `respond()` method decides whether the user is asking for a workout or a progress report. If the user asks for progress or history, the agent calls `progress_report()`. Otherwise, it generates a workout recommendation and saves the check-in to memory.

The `progress_report()` method summarizes saved history by reporting check-in count, average steps, average active minutes, and most common goal.

In [9]:
@dataclass
class UserProfile:
    name: str = 'Y. Powell'
    goal: str = 'maintain fitness'
    equipment: str = 'bodyweight'
    experience_level: str = 'beginner'
    limitations: str = 'none'
    history: List[Dict[str, Any]] = field(default_factory=list)
class FitBotWorkoutAgent:
    def __init__(self): self.profile = UserProfile()
    def update_profile(self, goal, equipment, experience_level, limitations):
        self.profile.goal=goal; self.profile.equipment=equipment; self.profile.experience_level=experience_level; self.profile.limitations=limitations
    def respond(self, message, goal, equipment, experience_level, limitations, steps, active_minutes, sedentary_minutes, workout_minutes):
        self.update_profile(goal, equipment, experience_level, limitations)
        if 'progress' in message.lower() or 'history' in message.lower(): return self.progress_report()
        workout = recommend_workout(goal, equipment, experience_level, limitations, int(steps), int(active_minutes), int(sedentary_minutes), int(workout_minutes))
        self.profile.history.append({'goal':goal, 'steps':int(steps), 'active_minutes':int(active_minutes), 'sedentary_minutes':int(sedentary_minutes), 'workout_minutes':int(workout_minutes)})
        return workout
    def progress_report(self):
        if not self.profile.history: return 'No workout history yet. Complete a few check-ins first.'
        df = pd.DataFrame(self.profile.history)
        return f"Progress Report\n\n- Check-ins saved: {len(df)}\n- Average steps: {int(df.steps.mean()):,}\n- Average active minutes: {int(df.active_minutes.mean())}\n- Most common goal: {df.goal.mode()[0]}\n\nNext step: repeat the workout that matched your goal best, then increase either time or one set next week."
fitbot_memory_agent = FitBotWorkoutAgent()


## Cell 17 — LangChain Tools Section

“This section connects my Python logic to LangChain. By wrapping FitBot functions as tools, a language model can decide which function to call based on the user’s request.”

This section introduces the LangChain tool layer. LangChain tools are Python functions wrapped in a format that an LLM agent can understand and call when needed.

Instead of only generating text, the LLM can call FitBot’s real functions, such as recommending a workout, predicting calories, or showing progress. This is what makes the system more agent-like.

## Cell 18 — Wrap FitBot Functions as LangChain Tools

“This cell makes FitBot LangChain-ready. It wraps the project’s functions as tools so an LLM agent can call them. I also used error handling so the notebook still works even if LangChain is not installed.”

This cell attempts to import LangChain’s `tool` decorator. If LangChain is installed, three FitBot functions are wrapped as tools: `recommend_fitbot_workout`, `predict_fitbot_calories`, and `show_fitbot_progress`.

The recommendation tool returns a personalized workout plan. The calorie prediction tool uses the trained machine learning model to estimate calories burned. The progress tool returns the saved memory report from the fallback agent.

The cell uses a try/except block so the notebook does not crash if LangChain is missing. If LangChain is available, `LANGCHAIN_TOOLS_READY` is set to True. If not, the notebook prints a helpful message and continues running with the fallback agent.

In [10]:
try:
    from langchain.tools import tool
    @tool
    def recommend_fitbot_workout(goal: str, equipment: str, experience_level: str, limitations: str, steps: int, active_minutes: int, sedentary_minutes: int, workout_minutes: int) -> str:
        """Recommend a personalized workout plan based on user goal, equipment, experience, limitations, and activity data."""
        return recommend_workout(goal, equipment, experience_level, limitations, steps, active_minutes, sedentary_minutes, workout_minutes)
    @tool
    def predict_fitbot_calories(steps: int, active_minutes: int, sedentary_minutes: int) -> str:
        """Predict estimated calories burned using the trained FitBot calorie model."""
        return f'Estimated calories burned: {predict_calories_from_activity(steps, active_minutes, sedentary_minutes)}'
    @tool
    def show_fitbot_progress() -> str:
        """Show the user's saved FitBot progress report."""
        return fitbot_memory_agent.progress_report()
    tools = [recommend_fitbot_workout, predict_fitbot_calories, show_fitbot_progress]
    LANGCHAIN_TOOLS_READY = True
    print('LangChain tools created successfully.')
except Exception as e:
    tools = []
    LANGCHAIN_TOOLS_READY = False
    print('LangChain tools not ready. Install LangChain first.')
    print(e)


LangChain tools created successfully.


## Cell 19 — LangChain Agent Section

“This section is optional because it requires an API key. I removed the key from the notebook for security, but the architecture is ready for LangChain tool-calling if a key is added.”

This section introduces the optional LangChain agent. The agent requires an OpenAI API key because it uses an LLM to interpret user requests and choose which tool to call.

The notebook is designed safely so the API key is not included in the code. The user can add their own key in Colab if they want to activate the LLM version.

## Cell 20 — Create LangChain LLM Agent

“This cell creates LLM-powered FitBot agent. The model can use FitBot’s tools based on the user’s request, but the notebook protects my API key by requiring it to be added separately in the environment.”

This cell checks whether LangChain tools are ready and whether an OpenAI API key exists in the environment. If both conditions are true, it imports `ChatOpenAI` and creates a LangChain agent.

The system prompt tells the agent how to behave as FitBot. It instructs the model to use the workout tool for workout requests, the calorie tool for calorie questions, and the progress tool for history questions. It also includes a safety instruction telling the assistant to keep advice practical and avoid medical advice.

If no API key is found, the cell prints a message explaining that the LangChain agent was not created. This prevents errors and keeps the notebook usable without paid API access.

In [11]:
# Add API key in Colab:
# os.environ['OPENAI_API_KEY'] = 'YOUR_API_KEY_HERE'
fitbot_langchain_agent = None
if LANGCHAIN_TOOLS_READY and os.environ.get('OPENAI_API_KEY'):
    try:
        from langchain_openai import ChatOpenAI
        from langchain.agents import create_agent
        llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)
        system_prompt = '''You are FitBot, an AI personal trainer agent. Use recommend_fitbot_workout for workout requests, predict_fitbot_calories for calorie questions, and show_fitbot_progress for progress/history questions. Keep advice practical and safe. Do not provide medical advice.'''
        fitbot_langchain_agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)
        print('LangChain FitBot agent created successfully.')
    except Exception as e:
        print('Could not create LangChain agent:', e)
else:
    print('LangChain agent not created. Add OPENAI_API_KEY to enable LLM tool-calling.')


LangChain agent not created. Add OPENAI_API_KEY to enable LLM tool-calling.


## Cell 21 — Fallback Agent Test Section

“This section tests the no-API version of the assistant. Even without an LLM, FitBot can still generate a personalized workout using the logic built earlier.”

This section tests the version of FitBot that works without an API key. This is important because it proves the project has a working core even if the optional LangChain agent is not activated.

The fallback agent still uses the same workout recommendation logic, calorie model, and memory system.

## Cell 22 — Test the Fallback FitBot Agent

“This cell is the first real demo. It sends FitBot a realistic user situation and prints a complete workout recommendation. It also saves the check-in to memory for future progress reports.”

This cell sends a sample request into the fallback memory agent. The sample user wants to lose weight, only has bodyweight equipment, is a beginner, has no limitations, and recorded 4,200 steps, 18 active minutes, 980 sedentary minutes, and 30 available workout minutes.

The agent processes the request, updates the user profile, generates a personalized workout, predicts calories, and saves the check-in to memory. The `print()` statement displays the assistant’s response so the result can be reviewed.

In [12]:
print(fitbot_memory_agent.respond('Recommend a workout', 'lose weight', 'bodyweight', 'beginner', 'none', 4200, 18, 980, 30))


Weight Loss Workout Recommendation

Estimated calories burned today: 1798
Steps: 4,200
Active minutes: 18
Workout time available: 30 minutes

Warm-Up:
- 5 minutes brisk walking or marching in place

Main Workout:
1. Brisk Walk — 1 sets, 20-30 min, rest as needed
2. Bodyweight Squat — 2-3 sets, 10-15, rest 60 sec
3. Incline Push-Up — 2-3 sets, 8-12, rest 60 sec
4. Forearm Plank — 2-3 sets, 20-40 sec, rest 45 sec

Cooldown:
- 5 minutes easy walking and stretching

Step Guidance: You are 5,800 steps below 10,000. Add a short walk later if you feel good.
Sedentary Guidance: Add 2-3 short movement breaks during the day.

Progression: If this workout feels easy for two sessions, add one set or 5 extra minutes next time.


## Cell 23 — LangChain Agent Test Section

“This section tests the LLM tool-calling version. If the API key is active, the user can type a natural language request and the LangChain agent can call the correct FitBot tool.”

This section tests the optional LangChain version of the assistant. It only runs if the LangChain agent was created successfully in the previous cells.

This test shows how a natural language prompt can be passed to an LLM agent, which then decides which FitBot tool to use.

## Cell 24 — Test the LangChain Tool-Calling Agent

“This cell demonstrates the LangChain architecture. The user writes a regular prompt, and the LLM agent can decide to call the FitBot workout tool instead of only guessing an answer.”

This cell checks whether `fitbot_langchain_agent` exists. If it does, the code sends a natural language workout request to the agent. The request includes the user’s goal, equipment, experience level, limitations, steps, active minutes, sedentary minutes, and workout time.

The LangChain agent should interpret the request, select the workout recommendation tool, call the function, and return the final response. If the agent is not active, the cell prints a message telling the user to either use the fallback agent or add an API key and rerun the setup.

In [13]:
if fitbot_langchain_agent is not None:
    response = fitbot_langchain_agent.invoke({'messages':[{'role':'user','content':'I want to lose weight. I only have bodyweight equipment. I am a beginner with no injuries. Today I got 4200 steps, 18 active minutes, and 980 sedentary minutes. I have 30 minutes. Please recommend a workout.'}]})
    print(response['messages'][-1].content)
else:
    print('LangChain agent is not active. Use Cell 10 or add an API key and rerun Cell 9.')


LangChain agent is not active. Use Cell 10 or add an API key and rerun Cell 9.


## Cell 25 — Optional Gradio Interface Section

“This section turns the notebook into a simple app. Gradio lets users enter their fitness information through a web interface and receive FitBot’s response.”

This section introduces the optional web interface. Gradio allows the project to become an interactive app instead of only a notebook.

The interface gives users text boxes, dropdowns, and number inputs so they can interact with FitBot more easily. This is useful for demonstrations because it makes the AI assistant look and feel like a real application.

## Cell 26 — Launch the Optional Gradio FitBot App

“This final cell creates the optional app version of FitBot. Instead of typing directly into code cells, users can interact with the AI assistant through dropdowns and text boxes.”

This cell tries to import Gradio and then defines a `fitbot_interface()` function. The interface function takes user inputs from the app, passes them into the fallback FitBot memory agent, and returns the generated workout response.

The Gradio interface includes inputs for the user message, fitness goal, equipment, experience level, limitations, steps, active minutes, sedentary minutes, and available workout time. The output is a large text box showing FitBot’s response.

The cell uses a try/except block so the notebook does not fail if Gradio is not installed. If Gradio is available, `demo.launch(share=True)` starts the app and creates a shareable link in Colab.

In [14]:
try:
    import gradio as gr
    def fitbot_interface(message, goal, equipment, experience_level, limitations, steps, active_minutes, sedentary_minutes, workout_minutes):
        return fitbot_memory_agent.respond(message, goal, equipment, experience_level, limitations, int(steps), int(active_minutes), int(sedentary_minutes), int(workout_minutes))
    demo = gr.Interface(fn=fitbot_interface, inputs=[gr.Textbox(label='Message', value='Recommend a workout'), gr.Dropdown(['lose weight','maintain fitness','gain strength','build endurance','recovery'], label='Goal', value='lose weight'), gr.Dropdown(['bodyweight','dumbbells','resistance bands','gym'], label='Equipment', value='bodyweight'), gr.Dropdown(['beginner','intermediate','advanced'], label='Experience Level', value='beginner'), gr.Textbox(label='Limitations or Injuries', value='none'), gr.Number(label='Steps Today', value=4200), gr.Number(label='Active Minutes', value=18), gr.Number(label='Sedentary Minutes', value=980), gr.Number(label='Workout Minutes Available', value=30)], outputs=gr.Textbox(label='FitBot Response', lines=24), title='FitBot LangChain-Ready AI Agent')
    demo.launch(share=True)
except Exception as e:
    print('Gradio is not installed. In Colab, run: !pip install gradio')
    print(e)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42b48ccd9402c5a9c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
